In [3]:
from huggingface_hub import hf_hub_download

years = [2024]

lamdap= {}

for yr in years:
    train_p = hf_hub_download(
        repo_id = "IQSeC-Lab/LAMDA",
        filename = f"Baseline/{yr}/{yr}_train.parquet",
        repo_type= "dataset"
    )
    test_p = hf_hub_download(   
        repo_id = "IQSeC-Lab/LAMDA",
        filename = f"Baseline/{yr}/{yr}_test.parquet",
        repo_type = "dataset"
    )

    lamdap[yr]={
        "train":train_p,
        "test": test_p
    }
    

/Users/angela/miniforge3/envs/aqua311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#%pip install pyarrow

In [4]:
import pandas as pd 

LAMDA_train_2024 = pd.read_parquet(lamdap[2024]["train"])
LAMDA_test_2024 = pd.read_parquet(lamdap[2024]["test"])


In [5]:
import numpy as np
import gc

LAMDA_train_2024 = pd.read_parquet(lamdap[2024]["train"])
print("2024: Shape:", LAMDA_train_2024.shape)

print("Columns:", LAMDA_train_2024.columns.tolist())
print("Counts (label):", LAMDA_train_2024.value_counts())
print("Percentages:", LAMDA_train_2024["label"].value_counts(normalize=True)*100)
print("Missing:", LAMDA_train_2024["label"].isna().sum())



2024: Shape: (38741, 4566)
Columns: ['hash', 'label', 'family', 'vt_count', 'year_month', 'feat_0', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6', 'feat_7', 'feat_8', 'feat_9', 'feat_10', 'feat_11', 'feat_12', 'feat_13', 'feat_14', 'feat_15', 'feat_16', 'feat_17', 'feat_18', 'feat_19', 'feat_20', 'feat_21', 'feat_22', 'feat_23', 'feat_24', 'feat_25', 'feat_26', 'feat_27', 'feat_28', 'feat_29', 'feat_30', 'feat_31', 'feat_32', 'feat_33', 'feat_34', 'feat_35', 'feat_36', 'feat_37', 'feat_38', 'feat_39', 'feat_40', 'feat_41', 'feat_42', 'feat_43', 'feat_44', 'feat_45', 'feat_46', 'feat_47', 'feat_48', 'feat_49', 'feat_50', 'feat_51', 'feat_52', 'feat_53', 'feat_54', 'feat_55', 'feat_56', 'feat_57', 'feat_58', 'feat_59', 'feat_60', 'feat_61', 'feat_62', 'feat_63', 'feat_64', 'feat_65', 'feat_66', 'feat_67', 'feat_68', 'feat_69', 'feat_70', 'feat_71', 'feat_72', 'feat_73', 'feat_74', 'feat_75', 'feat_76', 'feat_77', 'feat_78', 'feat_79', 'feat_80', 'feat_81', 'feat_82', 'feat_8

In [21]:
LAMDA_feature_cols = [
    col for col in LAMDA_train_2024.columns
    if col.startswith("feat_")
]
print(len(LAMDA_feature_cols))

4561


In [22]:
# time split
LAMDA_train_2024["year_month"] = pd.to_datetime(
    LAMDA_train_2024["year_month"]
)

train_m = (
    LAMDA_train_2024["year_month"] <= "2024-08-01"
)
val_m = (
    LAMDA_train_2024["year_month"].between(
        "2024-09-01", "2024-10-01"
    )
)
test_m = (
    LAMDA_train_2024["year_month"] >= "2024-11-01"
)


X_LAMDA_TRAIN = LAMDA_train_2024.loc[
    train_m, LAMDA_feature_cols
]
Y_LAMDA_TRAIN = LAMDA_train_2024.loc[
    train_m, "label"
]

X_LAMDA_VAL = LAMDA_train_2024.loc[
    val_m, LAMDA_feature_cols
]
Y_LAMDA_VAL = LAMDA_train_2024.loc[
    val_m, "label"
]

x_lamda_test_t = LAMDA_train_2024.loc[
    test_m, LAMDA_feature_cols
]

y_lamda_test_t = LAMDA_train_2024.loc[
    test_m, "label"
]

for n, x in [
    ("TRAIN", Y_LAMDA_TRAIN),
    ("VALIDATION", Y_LAMDA_VAL),
    ("TIME TEST", y_lamda_test_t)

]:
    print("rows", len(x))
    print("class counts:")
    print(x.value_counts())
    

    

rows 24896
class counts:
label
0    24369
1      527
Name: count, dtype: int64
rows 8354
class counts:
label
0    8288
1      66
Name: count, dtype: int64
rows 5491
class counts:
label
0    5449
1      42
Name: count, dtype: int64


BASELINE

In [23]:
# scaling
from sklearn.preprocessing import StandardScaler

scaling_L = StandardScaler()

X_LAMDA_TRAIN_SC = scaling_L.fit_transform(X_LAMDA_TRAIN)
X_LAMDA_VAL_SC = scaling_L.transform( X_LAMDA_VAL)
X_LAMDA_TEST_SC =scaling_L.transform(x_lamda_test_t)


In [24]:
#pytorch tensors
import torch 
from torch.utils.data import TensorDataset, DataLoader

X_LAMDA_TRAIN_T = torch.from_numpy(
    np.asarray(X_LAMDA_TRAIN_SC, dtype=np.float32)
)
Y_LAMDA_TRAIN_T = torch.from_numpy(
    Y_LAMDA_TRAIN.to_numpy(dtype=np.int64)
)

X_LAMDA_VAL_T = torch.from_numpy(
    np.asarray(X_LAMDA_VAL_SC , dtype=np.float32)
)
Y_LAMDA_VAL_T= torch.from_numpy(
    Y_LAMDA_VAL.to_numpy(dtype=np.int64)
)

X_LAMDA_TEST_T = torch.from_numpy(
    np.asarray(X_LAMDA_TEST_SC, dtype=np.float32)
)
Y_LAMDA_TEST_T = torch.from_numpy(
    y_lamda_test_t.to_numpy(dtype=np.int64)
)


In [26]:
train_dataset = TensorDataset(
    X_LAMDA_TRAIN_T,
    Y_LAMDA_TRAIN_T
)
val_dataset = TensorDataset(
    X_LAMDA_VAL_T,
    Y_LAMDA_VAL_T

)
test_dataset = TensorDataset(
    X_LAMDA_TEST_T,
    Y_LAMDA_TEST_T
)
train_dataload = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=False
)
val_dataload = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False
)
test_dataload = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

In [27]:
X_LAMDA_TEST = LAMDA_test_2024[LAMDA_feature_cols]
Y_LAMDA_TEST = LAMDA_test_2024["label"]

X_LAMDA_TEST_SC = scaling_L.transform(X_LAMDA_TEST)

X_LAMDA_TEST_T = torch.tensor(
    X_LAMDA_TEST_SC, dtype=torch.float32
)

Y_LAMDA_TEST_T = torch.tensor(
    Y_LAMDA_TEST.to_numpy(), dtype=torch.long
)

In [28]:
import torch
import torch.nn as nn
class LAMDATransformer(nn.Module):
    def __init__(
        s,
        input_dim = 4561,
        d_model = 64,
        nhead=4,
        num_layers=2,
        num_classes=2
    ):
        super().__init__()

        s.embedding = nn.Linear(input_dim, d_model)

        encoder_lay = nn.TransformerEncoderLayer(
            d_model = d_model,
            nhead=nhead,
            dim_feedforward=128,
            dropout=0.2,
            batch_first=True

        )

        s.transformer = nn.TransformerEncoder(
            encoder_lay,
            num_layers=num_layers
        )

        s.classifier = nn.Linear(d_model, num_classes)
    def forward(s,x):
        x = s.embedding(x)
        x = x.unsqueeze(1)
        x = s.transformer(x)
        x = x.squeeze(1)
        return s.classifier(x)
    

In [29]:
import torch.optim as optim
t_dev = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model_L = LAMDATransformer().to(t_dev)

class_c = torch.bincount(Y_LAMDA_TRAIN_T)
c_w = (
    len(Y_LAMDA_TRAIN_T)/
    (2.0*class_c.float())
)
c_w = c_w.to(t_dev)

criterionl = nn.CrossEntropyLoss(
    weight =c_w
)

optimizerL = optim.AdamW(
    model_L.parameters(),
    lr = 0.0001,
    weight_decay = 0.0001
)

In [30]:
# training
epoch_ = 5
for epoch in range(epoch_):
    model_L.train()
    tot_loss = 0
    for xx, yy in train_dataload:
        xx = xx.to(t_dev)
        yy = yy.to(t_dev)
        optimizerL.zero_grad()

        outputs = model_L(xx)
        lo = criterionl(outputs, yy)

        lo.backward()
        optimizerL.step()
        tot_loss += lo.item()

    averagelo = tot_loss / len(train_dataload)

    model_L.eval()
    val_loss = 0

    with torch.no_grad():
        for xx, yy in val_dataload:
            xx = xx.to(t_dev)
            yy = yy.to(t_dev)

            outputs = model_L(xx)
            lo = criterionl(outputs,yy)
            val_loss += lo.item()
    average_val_loss = val_loss / len(val_dataload)

    print(
      f" {epoch+1}/ {epoch_}"
      f" {averagelo}"
      f" {average_val_loss}"
    )

 1/ 5 0.380155758286009 0.404779518644015
 2/ 5 0.23224739198173797 0.3882744068449194
 3/ 5 0.17323847647224153 0.40939645785273926
 4/ 5 0.13467418197162298 0.4502849362113259
 5/ 5 0.11126336637808353 0.4951681686621724


In [31]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
import numpy as np
model_L.eval()

y_t = []
y_pd = []
y_prob = []

with torch.no_grad():
    for xx, yy in test_dataload:
        xx = xx.to(t_dev)

        outputs = model_L(xx)
        probabilities = torch.softmax(outputs, dim=1)
        pred = torch.argmax(outputs, dim=1)

        y_t.extend(yy.numpy())
        y_pd.extend(pred.cpu().numpy())
        y_prob.extend(probabilities[:,1].cpu().numpy())

y_t = np.array(y_t)
y_pd = np.array(y_pd)
y_prob = np.array(y_prob)

In [32]:
accuracy = accuracy_score(y_t,y_pd)
precision = precision_score(y_t,y_pd, zero_division=0)
recall = recall_score(y_t, y_pd, zero_division=0)
f1 = f1_score(y_t, y_pd, zero_division=0)
roc_auc = roc_auc_score(y_t, y_prob)
avg_auc = average_precision_score(y_t, y_prob)

print(f"Accuracy:", {accuracy})
print(f"Precision", {precision})
print(f"Recall", {recall})
print(f"F1 score:", {f1})
print("ROC_AUC:", {roc_auc})
print("Average precision:", {avg_auc})

Accuracy: {0.8887270078309962}
Precision {0.04038772213247173}
Recall {0.5952380952380952}
F1 score: {0.07564296520423601}
ROC_AUC: {0.8372440552657107}
Average precision: {0.12011594063740881}


In [33]:
model_L.eval()
lamda_tra_prob = []
lamda_tra_lab = []

with torch.no_grad():
    for xx, yy in train_dataload:
        xx = xx.to(t_dev)

        outputs = model_L(xx)
        probs = torch.softmax(outputs, dim=1)

        lamda_tra_prob.extend(probs.cpu().numpy())
        lamda_tra_lab.extend(yy.numpy())
lamda_train_prob = np.array(lamda_tra_prob)
lamda_train_lab = np.array(lamda_tra_lab)

print("PROB: ", lamda_train_prob.shape)
print("LABEL:", lamda_train_lab.shape)

PROB:  (24896, 2)
LABEL: (24896,)


In [34]:
from cleanlab.filter import find_label_issues

lamda_label_issue = find_label_issues(
    labels=lamda_train_lab,
    pred_probs=lamda_train_prob,
    return_indices_ranked_by="self_confidence"
)
print("Label issues:", len(lamda_label_issue))
print("Training samples:", len(lamda_train_lab))
print("Percentage: ", len(lamda_label_issue) / len(lamda_train_lab)*100)

python(86610) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86611) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86612) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86613) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86614) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Label issues: 172
Training samples: 24896
Percentage:  0.6908740359897172


In [35]:
# AUM
x_lamda_aum = torch.tensor(
    X_LAMDA_TRAIN_SC,
    dtype=torch.float32
)
y_lamda_aum = torch.tensor(
    Y_LAMDA_TRAIN.to_numpy(),
    dtype= torch.long
)
lamda_aum_dataset = TensorDataset(
    x_lamda_aum,
    y_lamda_aum
)

lamda_aum_load = DataLoader(
    lamda_aum_dataset,
    batch_size=256,
    shuffle=True
)
print(x_lamda_aum.shape)
print(y_lamda_aum.shape)


torch.Size([24896, 4561])
torch.Size([24896])


In [36]:
class AUMCompatibleMLP_LAMDA(nn.Module):
    def __init__(s, i_dim=4561, o_dim=3):
        super().__init__()
        
        s.o_dim = o_dim
    
        s.network = nn.Sequential(
            nn.Linear(i_dim, 64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,o_dim)
        )
    def forward(s, x, kwargs=None):
        return s.network(x)

In [37]:
import torch.optim as optim
aum_model_ = AUMCompatibleMLP_LAMDA(
    i_dim =4561,
    o_dim = 3
).to(t_dev)


aum_optimizer = optim.Adam(
    aum_model_.parameters(),
    lr = 0.001
)


In [38]:
import sys
import importlib
AQUA_path = "/Users/angela/aqua"
if AQUA_path not in sys.path:
    sys.path.insert(0, AQUA_path)

importlib.invalidate_caches()

import aqua
from aqua.configs import main_config
from aqua.data import Aqdata
from aqua.models.presets import AqNet
from aqua.models.cleaning_modules.aum.aum import AUM
print("done")

[nltk_data] Downloading package punkt to /Users/angela/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/angela/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


done


In [39]:
from aqua.models.cleaning_modules.aum.aum import AUM
import inspect
inspect.signature(AUM)

<Signature (model, optimizer)>

In [40]:
lamda_AQUA = Aqdata(
    X_LAMDA_TRAIN_SC.astype(np.float32),
    Y_LAMDA_TRAIN.to_numpy().astype(np.int64)

)
print(len(lamda_AQUA))

24896


In [41]:
aum_model_A = AqNet(
    model = aum_model_,
    output_dim=3,
    epochs=6,
    batch_size=256,
    lr=0.001,
    weighted_loss=True,
    device=str(t_dev),
    optimizer=aum_optimizer
)

In [42]:
aum_recorder_ = AUM(
    aum_model_A,
    aum_optimizer
)

In [43]:
print(inspect.signature(aum_recorder_.find_label_issues))

(data_aq, **kwargs)


In [44]:
import os
import tempfile

local_results_dir = os.path.join(
    tempfile.gettempdir(),
    "lamda_aum_results"
)

os.makedirs(local_results_dir, exist_ok=True)

main_config["results_dir"] = local_results_dir



In [45]:
lamda_aum_m = aum_recorder_.find_label_issues(
    lamda_AQUA,
    alpha=0.5
)

Epoch: 1/6: 100%|██████████| 98/98 [00:03<00:00, 27.95it/s, loss=0.707 (0.802)]
INFO:root:


Epoch: 2/6: 100%|██████████| 98/98 [00:03<00:00, 27.55it/s, loss=0.693 (0.717)]
INFO:root:


Epoch: 3/6: 100%|██████████| 98/98 [00:03<00:00, 26.99it/s, loss=0.758 (0.691)]
INFO:root:


Epoch: 4/6: 100%|██████████| 98/98 [00:03<00:00, 31.67it/s, loss=0.652 (0.666)]
INFO:root:


Epoch: 5/6: 100%|██████████| 98/98 [00:03<00:00, 28.60it/s, loss=0.601 (0.640)]
INFO:root:


Epoch: 6/6: 100%|██████████| 98/98 [00:03<00:00, 26.48it/s, loss=0.778 (0.614)]
INFO:root:


Epoch: 1/6: 100%|██████████| 98/98 [00:03<00:00, 26.22it/s, loss=0.715 (0.790)]
INFO:root:


Epoch: 2/6: 100%|██████████| 98/98 [00:03<00:00, 30.52it/s, loss=0.655 (0.705)]
INFO:root:


Epoch: 3/6: 100%|██████████| 98/98 [00:03<00:00, 30.06it/s, loss=0.648 (0.682)]
INFO:root:


Epoch: 4/6: 100%|██████████| 98/98 [00:03<00:00, 29.66it/s, loss=0.661 (0.661)]
INFO:root:


Epoch: 5/6: 100%|██████████| 98/98 [00:03<00:00, 27.51it/s, loss=0.690

In [46]:
print("Flagged samples:", lamda_aum_m.sum())
print("Total training samples:", len(lamda_aum_m))


Flagged samples: 12468
Total training samples: 24896


In [47]:
lamda_keep = ~np.asarray(lamda_aum_m, dtype=bool)
x_lamda_c = X_LAMDA_TRAIN_SC[lamda_keep]
y_lamda_c = Y_LAMDA_TRAIN.to_numpy()[lamda_keep]
print("Cleaned shape:", x_lamda_c.shape)
print("cleaned labels:", y_lamda_c.shape)

Cleaned shape: (12428, 4561)
cleaned labels: (12428,)


In [48]:
x_lamda_clean_torch = torch.tensor(
    x_lamda_c,
    dtype=torch.float32
)

y_lamda_clean_torch = torch.tensor(
    y_lamda_c, 
    dtype=torch.long
)
print("feature:", x_lamda_clean_torch.shape)
print("label:", y_lamda_clean_torch.shape)

feature: torch.Size([12428, 4561])
label: torch.Size([12428])


In [49]:
from torch.utils.data import TensorDataset, DataLoader

lamda_clean_dataset = TensorDataset(
    x_lamda_clean_torch,
    y_lamda_clean_torch
)

lamda_clean_dataloader = DataLoader(
    lamda_clean_dataset,
    batch_size=256,
    shuffle=True
)
print(len(lamda_clean_dataloader))

49


In [50]:
model_after = LAMDATransformer(
    input_dim=4561,
    d_model=64,
    nhead=4,
    num_layers=2,
    num_classes=2,

).to(t_dev)

optimizer_after = optim.Adam(
    model_after.parameters(),
    lr=0.001
)
criterion_after = nn.CrossEntropyLoss()

In [51]:
epoch = 5
for epochs in range(epoch):
    model_after.train()
    r_loss =0.0
    for input, labels in lamda_clean_dataloader:
        input = input.to(t_dev)
        labels = labels.to(t_dev)

        optimizer_after.zero_grad()

        outputs = model_after(input)
        l = criterion_after(outputs,labels)

        l.backward()
        optimizer_after.step()

        r_loss += l.item()

    print(
        f"EPOCH [{epochs+1}/{epoch}],"
        f"Training loss: {r_loss}/len(lamda_clean_dataloader)"
    )

EPOCH [1/5],Training loss: 4.523653382435441/len(lamda_clean_dataloader)
EPOCH [2/5],Training loss: 1.8666402269154787/len(lamda_clean_dataloader)
EPOCH [3/5],Training loss: 1.3127119184937328/len(lamda_clean_dataloader)
EPOCH [4/5],Training loss: 1.1329560917802155/len(lamda_clean_dataloader)
EPOCH [5/5],Training loss: 0.9562457645079121/len(lamda_clean_dataloader)


In [52]:
model_after.eval()
prediction_after = []
prob_after = []
true_label_after = []

with torch.no_grad():
    for inputs, labels in test_dataload:
        inputs = inputs.to(t_dev)
        outputs = model_after(inputs)
        prob = torch.softmax(outputs, dim=1)

        prediction_after.extend(
            torch.argmax(prob, dim=1).cpu().numpy()
        )

        prob_after.extend(
            prob[:,1].cpu().numpy()
        )
        true_label_after.extend(labels.numpy())

prediction_after = np.array(prediction_after)
prob_after = np.array(prob_after)
true_label_after = np.array(true_label_after)

print("Prediction:", prediction_after.shape)
print("Probability:", prob_after)


Prediction: (5491,)
Probability: [0.00117583 0.00040248 0.00275496 ... 0.00044999 0.00228    0.00063726]


In [53]:
accuracy_aft = accuracy_score(
    true_label_after, prediction_after
)
precision_aft = precision_score(
    true_label_after, prediction_after, zero_division=0
)
recall_aft = recall_score(
    true_label_after, prediction_after, zero_division=0
)
f1_after = f1_score(
    true_label_after, prediction_after, zero_division=0
)
roc_auc_after = roc_auc_score(
    true_label_after, prob_after
)
ap_after = average_precision_score(
    true_label_after, prob_after
)
print("Accuracy:", accuracy_aft)
print("Precision:", precision_aft)
print("Recall:", recall_aft)
print("F1:", f1_after)
print("ROC-AUC:", roc_auc_after)
print("Average Precision:", ap_after)

Accuracy: 0.9888909124021126
Precision: 0.22857142857142856
Recall: 0.19047619047619047
F1: 0.2077922077922078
ROC-AUC: 0.7183187828260318
Average Precision: 0.09292668051704821


In [59]:
import pandas as pd
scores_before = pd.DataFrame({
    "row_id": [f"LAMDA_TEST_{i}" for i in range(len(y_t))],
    "value": "baseline",
    "column":"Transformer",
    "label": y_t,
    "score":y_prob,
    "threshold":0.5
})


In [60]:
scores_after = pd.DataFrame({
    "row_id": [f"LAMDA_TEST_{i}" for i in
range(len(true_label_after))],
    "value":"after_aqua",
    "column": "Transformer",
    "label": true_label_after,
    "score": prob_after,
    "threshold":0.5
})

In [63]:
scores_LAMDA = pd.concat(
    [scores_before, scores_after],
    ignore_index=True
)

scores_LAMDA.to_csv(
    "LAMDA_scores.csv",
    index=False

)


In [81]:
LAMDA_split = pd.DataFrame({
    "row_id":(
        [f"LAMDA_TRAIN_{i}" for i in range (len(Y_LAMDA_TRAIN_T))]+
        [f"LAMDA_VAL_{i}" for i in range(len(Y_LAMDA_VAL_T))]+
        [f"LAMDA_TEST_{i}" for i in range(len(y_t))]  
),
"dataset":["LAMDA"]*(
    len(Y_LAMDA_TRAIN_T)+
    len(Y_LAMDA_VAL_T)+
    len(y_t)
),
"split": (
    ["train"]*len(Y_LAMDA_TRAIN_T)+
    ["validation"]*len(Y_LAMDA_VAL_T)+
    ["test"]*len(y_t)
),
"label": np.concatenate([
    np.asarray(Y_LAMDA_TRAIN_T).reshape(-1),
    np.asarray(Y_LAMDA_VAL_T).reshape(-1),
    np.asarray(y_t).reshape(-1)
])
})
LAMDA_split.to_csv("LAMDA_split.csv",index=False)


In [94]:
fp_ = np.where((y_t==0) & (y_pd ==1))[0]
fn_ = np.where((y_t ==1) & (y_pd==0))[0]

fp_sample = fp_[:10]
fn_sample = fn_[:10]

lamda_errors = pd.DataFrame({
    "row_id": (
        [f"LAMDA_TEST_{i}" for i in fp_sample]+
        [f"LAMDA_TEST_{i}" for i in fn_sample]
    ),
    "error_type":(
        ["false_positive"]* len(fp_sample)+
        ["false_negative"]* len(fn_sample)
    ),
    "true_label": np.concatenate([
        y_t[fp_sample], y_t[fn_sample]
    ]),
    "predicted_label": np.concatenate([
        y_pd[fp_sample], y_pd[fn_sample]
    ]),
    "score": np.concatenate([
        y_prob[fp_sample], y_prob[fn_sample]
    ])
    

})
lamda_errors["value"] = "baseline"
lamda_errors["column"] = "Transformer"
lamda_errors["category"] = np.where(
    lamda_errors["error_type"] == "false_positive",
    "benign_predicted_as_malware",
    "malware_predicted_as_benign"
)

lamda_errors.to_csv("LAMDA_errors.csv", index=False)

In [89]:
import subprocess
import sys
ssc = "/Users/angela/Desktop/check_final_checkpoint.py"
rrs_LAMDA = subprocess.run(
    [
        sys.executable,
        ssc,
        "--split", "/Users/angela/LAMDA_split.csv",
        "--scores", "/Users/angela/LAMDA_scores.csv",
        "--errors", "/Users/angela/LAMDA_errors.csv",
        "--metric", "recall",
        "--reference", "baseline",
        "--direction", "higher"
    ],
    capture_output = True,
    text=True
)
print(rrs_LAMDA.stdout)
print("Errors:")
print(rrs_LAMDA.stderr)

python(88122) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


split.csv: 38741 rows. scores.csv: 10982 rows, 2 value(s), 1 column(s).
Values: baseline, after_aqua

Rows per dataset and split
--------------------------
dataset      split  rows  positive  negative  positive %
  LAMDA      train 24896       527     24369        2.12
  LAMDA validation  8354        66      8288        0.79
  LAMDA       test  5491        42      5449        0.76

Metrics per value and column (positive when score >= the threshold)
-------------------------------------------------------------------
Each value is followed by its 95% bootstrap interval in brackets.

Value: baseline   Column: Transformer
  threshold                 0.5
  test rows                 5491
  positive rows             42 (0.76%)
  negative rows             5449 (99.24%)
  bootstrap resamples       1000
  random-scorer AP          0.0076 (equals the positive share)
  random-scorer AUROC       0.5000
  precision                 0.0404 [0.0250, 0.0568]
  recall                    0.5952 [0.4418, 0

In [ ]:
import subprocess
import sys
ssc = "/Users/angela/Desktop/check_final_checkpoint.py"
rrs_LAMDA = subprocess.run(
    [
        sys.executable,
        ssc,
        "--split", "/Users/angela/LAMDA_split.csv",
        "--scores", "/Users/angela/LAMDA_scores.csv",
        "--errors", "/Users/angela/LAMDA_errors.csv",
        "--metric", "recall",
        "--reference", "baseline",
        "--direction", "higher"
    ],
    capture_output = True,
    text=True
)
print(rrs_LAMDA.stdout)
print("Errors:")
print(rrs_LAMDA.stderr)



python(88210) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


split.csv: 38741 rows. scores.csv: 10982 rows, 2 value(s), 1 column(s).
Values: baseline, after_aqua

Rows per dataset and split
--------------------------
dataset      split  rows  positive  negative  positive %
  LAMDA      train 24896       527     24369        2.12
  LAMDA validation  8354        66      8288        0.79
  LAMDA       test  5491        42      5449        0.76

Metrics per value and column (positive when score >= the threshold)
-------------------------------------------------------------------
Each value is followed by its 95% bootstrap interval in brackets.

Value: baseline   Column: Transformer
  threshold                 0.5
  test rows                 5491
  positive rows             42 (0.76%)
  negative rows             5449 (99.24%)
  bootstrap resamples       1000
  random-scorer AP          0.0076 (equals the positive share)
  random-scorer AUROC       0.5000
  precision                 0.0404 [0.0250, 0.0568]
  recall                    0.5952 [0.4418, 0

In [97]:
with open("LAMDA_final_checkpoint_results.txt", "w") as f:
    f.write(rrs_LAMDA.stdout)
    f.write("\n\nErrors:\n")
    f.write(rrs_LAMDA.stderr)
    